# Notebook 52 — Piecewise Meta-Law and Regime Gating

Notebook 51 suggested the universality boundary is better understood as **regime-structured** rather than as a single smooth global map.
This notebook upgrades the coefficient meta-law with:

1. **Global meta-law**
2. **Piecewise meta-law**
   - cluster regimes in coefficient space
   - fit one predictor per cluster
3. **Soft-gated meta-law**
   - predict cluster membership from metadata
   - blend cluster-wise predictors

## Fixed governing template

\[
g(r,c;\beta)=\beta_0+\beta_1 c+\beta_2 r+\beta_3 c^3+\beta_4 r^2+\beta_5 r c^2
\]

## Main goals

1. Learn regime clusters in coefficient space.
2. Compare global vs piecewise vs soft-gated meta-laws on harder blocks.
3. Visualize regime geometry and cluster usage.
4. Check whether clustering repairs boundary failures.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.cluster import KMeans

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Fixed template and per-regime coefficient dataset

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(["system", "task", "forcing_mode", "k", "flow_mode"]):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
display(coef_df.head())

In [ ]:
# Metadata matrix

meta_df = coef_df[["regime_id", "system", "task", "forcing_mode", "flow_mode", "k"]].copy()
X_base = pd.get_dummies(meta_df[["system", "task", "forcing_mode", "flow_mode"]], drop_first=False)
X_base["k"] = coef_df["k"].astype(float).values
X_base["k2"] = coef_df["k"].astype(float).values**2
Y_coef = coef_df[COEF_COLS].copy()

display(X_base.head())
display(Y_coef.head())

In [ ]:
# Global meta-law helpers

def fit_predict_linear(X_train, Y_train, X_test):
    model = LinearRegression()
    model.fit(np.asarray(X_train, float), np.asarray(Y_train, float))
    return model.predict(np.asarray(X_test, float))

def fit_predict_ridge(X_train, Y_train, X_test):
    model = Ridge(alpha=1.0)
    model.fit(np.asarray(X_train, float), np.asarray(Y_train, float))
    return model.predict(np.asarray(X_test, float))

def fit_predict_latent_linear(X_train, Y_train, X_test):
    scaler = StandardScaler()
    Ytr_std = scaler.fit_transform(np.asarray(Y_train, float))
    pca = PCA(n_components=2, random_state=42)
    Ztr = pca.fit_transform(Ytr_std)
    model = LinearRegression()
    model.fit(np.asarray(X_train, float), Ztr)
    zhat = model.predict(np.asarray(X_test, float))
    yhat_std = pca.inverse_transform(np.atleast_2d(zhat))
    return scaler.inverse_transform(yhat_std)

def fit_predict_latent_ridge(X_train, Y_train, X_test):
    scaler = StandardScaler()
    Ytr_std = scaler.fit_transform(np.asarray(Y_train, float))
    pca = PCA(n_components=2, random_state=42)
    Ztr = pca.fit_transform(Ytr_std)
    model = Ridge(alpha=1.0)
    model.fit(np.asarray(X_train, float), Ztr)
    zhat = model.predict(np.asarray(X_test, float))
    yhat_std = pca.inverse_transform(np.atleast_2d(zhat))
    return scaler.inverse_transform(yhat_std)

beta_shared = Y_coef.to_numpy(dtype=float).mean(axis=0)

## Learn regime clusters in coefficient space

In [ ]:
coef_mat = Y_coef.to_numpy(dtype=float)
coef_std = StandardScaler().fit_transform(coef_mat)
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(coef_std)

coef_df["pc1"] = Z[:, 0]
coef_df["pc2"] = Z[:, 1]

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
coef_df["cluster"] = kmeans.fit_predict(Z)

display(coef_df[["regime_id", "system", "task", "forcing_mode", "k", "flow_mode", "pc1", "pc2", "cluster"]].head())

In [ ]:
plt.figure(figsize=(8, 5))
for cl in sorted(coef_df["cluster"].unique()):
    sub = coef_df[coef_df["cluster"] == cl]
    plt.scatter(sub["pc1"], sub["pc2"], label=f"cluster {cl}")
    for _, r in sub.iterrows():
        plt.text(r["pc1"], r["pc2"], r["forcing_mode"], fontsize=7, alpha=0.6)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Regime clusters in coefficient space")
plt.legend()
plt.tight_layout()
plt.show()

## Piecewise and soft-gated predictors

In [ ]:
def fit_predict_piecewise_linear(train_df, X_train, Y_train, X_test_row):
    preds = {}
    for cl in sorted(train_df["cluster"].unique()):
        mask = train_df["cluster"] == cl
        if mask.sum() < 2:
            continue
        preds[cl] = fit_predict_linear(X_train.loc[mask], Y_train.loc[mask], X_test_row)[0]

    clf = LogisticRegression(max_iter=2000, multi_class="auto")
    clf.fit(np.asarray(X_train, float), train_df["cluster"].to_numpy())
    cl_pred = int(clf.predict(np.asarray(X_test_row, float))[0])

    if cl_pred in preds:
        return preds[cl_pred], cl_pred

    fallback_cl = sorted(preds.keys())[0]
    return preds[fallback_cl], fallback_cl

def fit_predict_soft_gated(train_df, X_train, Y_train, X_test_row):
    local_preds = []
    active_clusters = []

    for cl in sorted(train_df["cluster"].unique()):
        mask = train_df["cluster"] == cl
        if mask.sum() < 2:
            continue
        local_preds.append(fit_predict_linear(X_train.loc[mask], Y_train.loc[mask], X_test_row)[0])
        active_clusters.append(cl)

    clf = LogisticRegression(max_iter=2000, multi_class="auto")
    clf.fit(np.asarray(X_train, float), train_df["cluster"].to_numpy())
    proba = clf.predict_proba(np.asarray(X_test_row, float))[0]

    weight_map = {cl: 0.0 for cl in active_clusters}
    for cl, p in zip(clf.classes_, proba):
        if cl in weight_map:
            weight_map[cl] = p

    weights = np.array([weight_map[cl] for cl in active_clusters], dtype=float)
    if weights.sum() <= 0:
        weights = np.ones(len(active_clusters)) / len(active_clusters)
    else:
        weights = weights / weights.sum()

    yhat = np.sum([w * pred for w, pred in zip(weights, local_preds)], axis=0)
    return yhat, dict(zip(active_clusters, weights))

## Compare global vs piecewise vs soft-gated on harder blocks

In [ ]:
hard_block_masks = {
    "k_extrapolate_high": coef_df["k"].astype(float) == 7.0,
    "k_extrapolate_low": coef_df["k"].astype(float) == 3.0,
    "system_holdout::entropy": coef_df["system"].astype(str) == "entropy",
    "system_holdout::unevenness": coef_df["system"].astype(str) == "unevenness",
}

compare_rows = []

for block_name, test_mask in hard_block_masks.items():
    train_mask = ~test_mask

    X_train = X_base.loc[train_mask].reset_index(drop=True)
    X_test = X_base.loc[test_mask].reset_index(drop=True)
    Y_train = Y_coef.loc[train_mask].reset_index(drop=True)
    Y_test = Y_coef.loc[test_mask].reset_index(drop=True)
    train_meta = coef_df.loc[train_mask].reset_index(drop=True)
    test_meta = coef_df.loc[test_mask].reset_index(drop=True)

    for i in range(len(test_meta)):
        regime_id = test_meta.iloc[i]["regime_id"]
        beta_true = Y_test.iloc[i].to_numpy(dtype=float)
        x_te = X_test.iloc[[i]]

        global_candidates = {
            "global_linear": fit_predict_linear(X_train, Y_train, x_te)[0],
            "global_ridge": fit_predict_ridge(X_train, Y_train, x_te)[0],
            "global_latent_linear": fit_predict_latent_linear(X_train, Y_train, x_te)[0],
            "global_latent_ridge": fit_predict_latent_ridge(X_train, Y_train, x_te)[0],
        }
        best_global_name = min(global_candidates, key=lambda k: math.sqrt(mean_squared_error(beta_true, global_candidates[k])))
        beta_global = global_candidates[best_global_name]

        beta_piece, cl_pred = fit_predict_piecewise_linear(train_meta, X_train, Y_train, x_te)
        beta_soft, weight_dict = fit_predict_soft_gated(train_meta, X_train, Y_train, x_te)

        sub = regime_subsets[regime_id]

        compare_rows.append({
            "block": block_name,
            "regime_id": regime_id,
            "best_global_model": best_global_name,

            "coef_rmse_global": math.sqrt(mean_squared_error(beta_true, beta_global)),
            "coef_rmse_piecewise": math.sqrt(mean_squared_error(beta_true, beta_piece)),
            "coef_rmse_soft": math.sqrt(mean_squared_error(beta_true, beta_soft)),

            "traj_rmse_global": trajectory_gap(sub, beta_true, beta_global),
            "traj_rmse_piecewise": trajectory_gap(sub, beta_true, beta_piece),
            "traj_rmse_soft": trajectory_gap(sub, beta_true, beta_soft),

            "cluster_pred_piecewise": cl_pred,
            "soft_gate_weights": str(weight_dict),
        })

compare_df = pd.DataFrame(compare_rows)
display(compare_df.head())

In [ ]:
summary_df = compare_df.groupby("block")[[
    "coef_rmse_global", "coef_rmse_piecewise", "coef_rmse_soft",
    "traj_rmse_global", "traj_rmse_piecewise", "traj_rmse_soft"
]].mean().reset_index()

display(summary_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(summary_df))
w = 0.26

axes[0].bar(x - w, summary_df["coef_rmse_global"], width=w, label="global")
axes[0].bar(x, summary_df["coef_rmse_piecewise"], width=w, label="piecewise")
axes[0].bar(x + w, summary_df["coef_rmse_soft"], width=w, label="soft-gated")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[0].set_title("Coefficient RMSE by harder block")
axes[0].legend()

axes[1].bar(x - w, summary_df["traj_rmse_global"], width=w, label="global")
axes[1].bar(x, summary_df["traj_rmse_piecewise"], width=w, label="piecewise")
axes[1].bar(x + w, summary_df["traj_rmse_soft"], width=w, label="soft-gated")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[1].set_title("Trajectory RMSE by harder block")
axes[1].legend()

plt.tight_layout()
plt.show()

## Gating behavior

In [ ]:
piece_counts = compare_df.groupby(["block", "cluster_pred_piecewise"]).size().reset_index(name="count")
display(piece_counts)

pivot = piece_counts.pivot(index="cluster_pred_piecewise", columns="block", values="count").fillna(0)
pivot.plot(kind="bar", figsize=(10, 5))
plt.ylabel("count")
plt.title("Piecewise cluster selection by harder block")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Representative regime comparison

In [ ]:
rep_idx = int(np.argmax(compare_df["traj_rmse_global"] - compare_df["traj_rmse_soft"]))
rep = compare_df.iloc[rep_idx]
regime_id = rep["regime_id"]

test_mask = coef_df["regime_id"] == regime_id
train_mask = ~test_mask
X_train = X_base.loc[train_mask].reset_index(drop=True)
X_test = X_base.loc[test_mask].reset_index(drop=True)
Y_train = Y_coef.loc[train_mask].reset_index(drop=True)
Y_test = Y_coef.loc[test_mask].reset_index(drop=True)
train_meta = coef_df.loc[train_mask].reset_index(drop=True)

beta_true = Y_test.iloc[0].to_numpy(dtype=float)
beta_global = fit_predict_linear(X_train, Y_train, X_test)[0]
beta_piece, cl_pred = fit_predict_piecewise_linear(train_meta, X_train, Y_train, X_test)
beta_soft, weight_dict = fit_predict_soft_gated(train_meta, X_train, Y_train, X_test)

sub = regime_subsets[regime_id]
cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
rmin, rmax = sub["residual"].min(), sub["residual"].max()
cgrid = np.linspace(cmin, cmax, 45)
r0s = np.linspace(np.quantile(sub["residual"], 0.1), np.quantile(sub["residual"], 0.9), 8)
flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

def integrate_beta(beta, r0):
    r = float(np.clip(r0, rmin, rmax))
    traj = [r]
    for j in range(len(cgrid) - 1):
        c = cgrid[j]
        dc = float(cgrid[j+1] - cgrid[j])
        x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
        g = float(np.clip(x @ beta, -flow_cap, flow_cap))
        r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
        traj.append(r)
    return np.array(traj)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
titles = ["Global linear", f"Piecewise (cluster {cl_pred})", "Soft-gated", "Regime-specific"]
betas = [beta_global, beta_piece, beta_soft, beta_true]

for ax, title, beta in zip(axes, titles, betas):
    for r0 in r0s:
        ax.plot(cgrid, integrate_beta(beta, r0))
    ax.set_title(title)
    ax.set_xlabel("condition coordinate c")
axes[0].set_ylabel("residual")
fig.suptitle(f"Representative regime: {regime_id}", y=1.03)
plt.tight_layout()
plt.show()

print("Soft gate weights:", weight_dict)

## Final verdicts

In [ ]:
def verdict_from_summary(row):
    if row["traj_rmse_soft"] < row["traj_rmse_global"] and row["traj_rmse_piecewise"] < row["traj_rmse_global"]:
        if row["traj_rmse_soft"] <= row["traj_rmse_piecewise"]:
            return "soft gating best"
        return "piecewise best"
    if row["traj_rmse_soft"] < row["traj_rmse_global"]:
        return "soft gating helps"
    if row["traj_rmse_piecewise"] < row["traj_rmse_global"]:
        return "piecewise helps"
    return "global model sufficient"

summary_df["verdict"] = summary_df.apply(verdict_from_summary, axis=1)
display(summary_df)

## Working conclusion

Notebook 52 tests whether the universality boundary from Notebook 51 is better handled by:
- a single global law
- piecewise meta-laws over regime clusters
- soft gating across clusters

A strong result is:
- harder blocks improve under clustering or soft gating
- cluster structure aligns with coefficient-space geometry
- gating provides smoother transitions than hard piecewise selection

If that pattern holds on your real exports, the next notebook should be:

**Notebook 53 — adaptive template selection within regime clusters**